In [ ]:
import sys
sys.path.append("../..")
sys.path.append("../../src")
import os
import pickle
import numpy as np
from causaldag import unknown_target_igsp
from causaldag import partial_correlation_test, MemoizedCI_Tester, partial_correlation_suffstat
from causaldag import MemoizedInvarianceTester, gauss_invariance_test, gauss_invariance_suffstat
from src.tools.metric import get_compared_components, get_skeleton, metric_skeleton_level, metric_cpdag_level

In [ ]:
def metric_target_level_for_utigsp(pred, targ):
    """ P / R / F1 (I-TARGET Level) """
    TP , TP_FP, TP_FN = 0, 0, 0
    for set_pred, set_targ in zip(pred, targ):
        TP += len(set_pred.intersection(set_targ))
        TP_FP += len(set_pred)
        TP_FN += len(set_targ)
    precision = TP / max(TP_FP, 1)
    recall = TP / max(TP_FN, 1)
    f1 = 2 * precision * recall / (precision + recall) if precision + recall != 0 else 0
    pred_iden_edges_num, targ_iden_edges_num = TP_FP, TP_FN
    return {'p':round(precision,2), 'r':round(recall,2), 'f1':round(f1,2), '#pred_iden_edges':int(pred_iden_edges_num), '#targ_iden_edges':int(targ_iden_edges_num)}

In [ ]:
exp_name = 'exp_200'
intervention_size_list = [1, 1, 2, 2, 4, 5, 6, 7, 7, 10, 11, 14, 15, 22]
os.makedirs(f'../../exps_of_result/ut-igsp/{exp_name}/know', exist_ok=True)
os.makedirs(f'../../baselines/exps_of_result/ut-igsp/{exp_name}/unknow', exist_ok=True)


for idx, benchmark_name in enumerate(['01earthquake', '02survey', '03asia', '04sachs',  '05child', '06insurance', '07water', '08mildew', '09alarm', '10barley', '11hailfinder', '12hepar2', '13win95pts', '14pathfinder']):
    print(f"{'-'*60} {benchmark_name} {'-'*60} \n")
    intervention_size = intervention_size_list[idx]
    # ground truth of I-SKELETON / I-TARGETS / I-CPDAG
    aug_graph_path = f'../../datasets/experiment/original/{exp_name}/aug_graphs/{benchmark_name}_aug_graph.txt'
    raw_dataset_path = f'../../datasets/experiment/original/{exp_name}/raw_samples/{benchmark_name}_raw_dataset.pkl'
    int_targets_path = f'../../datasets/experiment/original/{exp_name}/int_targets/{benchmark_name}_int_targets.pkl'
    targ_I_SKELETON, targ_I_TARGETS, targ_I_CPDAG = get_compared_components(aug_graph_path, intervention_size, real=True)
    
    try:
        know_pred_graph_path = f'../../baselines/exps_of_result/ut-igsp/{exp_name}/know/{benchmark_name}_aug_graph.txt'
        unknow_pred_graph_path = f'../../baselines/exps_of_result/ut-igsp/{exp_name}/unknow/{benchmark_name}_aug_graph.txt'

        with open(raw_dataset_path, 'rb') as f:
            data_list = pickle.load(f)
        with open(int_targets_path, 'rb') as f:
            know_targets_list = pickle.load(f)
        
        obs_samples, iv_samples_list = data_list[0], data_list[1:]
        nodes = set(range(obs_samples.shape[1]))
        obs_suffstat = partial_correlation_suffstat(obs_samples)
        ci_tester = MemoizedCI_Tester(partial_correlation_test, obs_suffstat, alpha=1e-3)
        invariance_suffstat = gauss_invariance_suffstat(obs_samples, iv_samples_list)
        invariance_tester = MemoizedInvarianceTester(gauss_invariance_test, invariance_suffstat, alpha=1e-3)
        
        unknow_setting_list = [dict(known_interventions=[]) for _ in know_targets_list[1:]]
        know_setting_list = [dict(known_interventions=targets) for targets in know_targets_list[1:]]
        
        pred_I_CPDAG_unknow, pred_I_TARGETS_unknow = unknown_target_igsp(unknow_setting_list, nodes, ci_tester, invariance_tester)
        pred_I_CPDAG_know, _ = unknown_target_igsp(know_setting_list, nodes, ci_tester, invariance_tester)
        
        pred_I_CPDAG_unknow, pred_I_CPDAG_know = pred_I_CPDAG_unknow.to_amat()[0], pred_I_CPDAG_know.to_amat()[0]

        with open(unknow_pred_graph_path, 'wb') as f:
            np.savetxt(f, pred_I_CPDAG_unknow, fmt='%i')

        with open(know_pred_graph_path, 'wb') as f:
            np.savetxt(f, pred_I_CPDAG_know, fmt='%i')
        
        pred_I_SKELETON_unknow = get_skeleton(pred_I_CPDAG_unknow)
        mt_skeleton_unknow = metric_skeleton_level(pred_I_SKELETON_unknow, targ_I_SKELETON)
        mt_target_unknow = metric_target_level_for_utigsp(pred_I_TARGETS_unknow, list(map(set, know_targets_list[1:])))
        mt_cpdag_unknow = metric_cpdag_level(pred_I_CPDAG_unknow, targ_I_CPDAG)
        print(f'unknow performance: \n skeleton:{mt_skeleton_unknow} \n targets:{mt_target_unknow} \n cpdag:{mt_cpdag_unknow} \n')

        pred_I_SKELETON_know = get_skeleton(pred_I_CPDAG_know)
        mt_skeleton_know = metric_skeleton_level(pred_I_SKELETON_know, targ_I_SKELETON)
        mt_cpdag_know = metric_cpdag_level(pred_I_CPDAG_know, targ_I_CPDAG)
        print(f'know performance: \n skeleton:{mt_skeleton_know} \n  cpdag:{mt_cpdag_know} \n')
    except:
        print(f'pass {benchmark_name}\n')

In [ ]:
exp_name = 'exp_1_1'
intervention_size_list = [1, 1, 2, 2, 4, 5, 6, 7, 7, 10, 11, 14, 15, 22]
os.makedirs(f'../../exps_of_result/ut-igsp/{exp_name}/know', exist_ok=True)
os.makedirs(f'../../baselines/exps_of_result/ut-igsp/{exp_name}/unknow', exist_ok=True)


for idx, benchmark_name in enumerate(['01earthquake', '02survey', '03asia', '04sachs',  '05child', '06insurance', '07water', '08mildew', '09alarm', '10barley', '11hailfinder', '12hepar2', '13win95pts', '14pathfinder']):
    if idx==0 or idx>=8:
        continue
    print(f"{'-'*60} {benchmark_name} {'-'*60} \n")
    intervention_size = intervention_size_list[idx]
    # ground truth of I-SKELETON / I-TARGETS / I-CPDAG
    aug_graph_path = f'../../datasets/experiment/original/{exp_name}/aug_graphs/{benchmark_name}_aug_graph.txt'
    raw_dataset_path = f'../../datasets/experiment/original/{exp_name}/raw_samples/{benchmark_name}_raw_dataset.pkl'
    int_targets_path = f'../../datasets/experiment/original/{exp_name}/int_targets/{benchmark_name}_int_targets.pkl'
    targ_I_SKELETON, targ_I_TARGETS, targ_I_CPDAG = get_compared_components(aug_graph_path, intervention_size, real=True)
    
    try:
        know_pred_graph_path = f'../../baselines/exps_of_result/ut-igsp/{exp_name}/know/{benchmark_name}_aug_graph.txt'
        unknow_pred_graph_path = f'../../baselines/exps_of_result/ut-igsp/{exp_name}/unknow/{benchmark_name}_aug_graph.txt'

        with open(raw_dataset_path, 'rb') as f:
            data_list = pickle.load(f)
        with open(int_targets_path, 'rb') as f:
            know_targets_list = pickle.load(f)
        
        obs_samples, iv_samples_list = data_list[0], data_list[1:]
        nodes = set(range(obs_samples.shape[1]))
        obs_suffstat = partial_correlation_suffstat(obs_samples)
        ci_tester = MemoizedCI_Tester(partial_correlation_test, obs_suffstat, alpha=1e-3)
        invariance_suffstat = gauss_invariance_suffstat(obs_samples, iv_samples_list)
        invariance_tester = MemoizedInvarianceTester(gauss_invariance_test, invariance_suffstat, alpha=1e-3)
        
        unknow_setting_list = [dict(known_interventions=[]) for _ in know_targets_list[1:]]
        know_setting_list = [dict(known_interventions=targets) for targets in know_targets_list[1:]]
        
        pred_I_CPDAG_unknow, pred_I_TARGETS_unknow = unknown_target_igsp(unknow_setting_list, nodes, ci_tester, invariance_tester)
        pred_I_CPDAG_know, _ = unknown_target_igsp(know_setting_list, nodes, ci_tester, invariance_tester)
        
        pred_I_CPDAG_unknow, pred_I_CPDAG_know = pred_I_CPDAG_unknow.to_amat()[0], pred_I_CPDAG_know.to_amat()[0]

        with open(unknow_pred_graph_path, 'wb') as f:
            np.savetxt(f, pred_I_CPDAG_unknow, fmt='%i')

        with open(know_pred_graph_path, 'wb') as f:
            np.savetxt(f, pred_I_CPDAG_know, fmt='%i')
        
        pred_I_SKELETON_unknow = get_skeleton(pred_I_CPDAG_unknow)
        mt_skeleton_unknow = metric_skeleton_level(pred_I_SKELETON_unknow, targ_I_SKELETON)
        mt_target_unknow = metric_target_level_for_utigsp(pred_I_TARGETS_unknow, list(map(set, know_targets_list[1:])))
        mt_cpdag_unknow = metric_cpdag_level(pred_I_CPDAG_unknow, targ_I_CPDAG)
        print(f'unknow performance: \n skeleton:{mt_skeleton_unknow} \n targets:{mt_target_unknow} \n cpdag:{mt_cpdag_unknow} \n')

        pred_I_SKELETON_know = get_skeleton(pred_I_CPDAG_know)
        mt_skeleton_know = metric_skeleton_level(pred_I_SKELETON_know, targ_I_SKELETON)
        mt_cpdag_know = metric_cpdag_level(pred_I_CPDAG_know, targ_I_CPDAG)
        print(f'know performance: \n skeleton:{mt_skeleton_know} \n  cpdag:{mt_cpdag_know} \n')
    except:
        print(f'pass {benchmark_name}\n')

In [ ]:
exp_name = 'exp_1_2'
intervention_size_list = [1, 1, 2, 2, 4, 5, 6, 7, 7, 10, 11, 14, 15, 22]
os.makedirs(f'../../exps_of_result/ut-igsp/{exp_name}/know', exist_ok=True)
os.makedirs(f'../../baselines/exps_of_result/ut-igsp/{exp_name}/unknow', exist_ok=True)


for idx, benchmark_name in enumerate(['01earthquake', '02survey', '03asia', '04sachs',  '05child', '06insurance', '07water', '08mildew', '09alarm', '10barley', '11hailfinder', '12hepar2', '13win95pts', '14pathfinder']):
    if idx==0 or idx>=8:
        continue
    print(f"{'-'*60} {benchmark_name} {'-'*60} \n")
    intervention_size = intervention_size_list[idx]
    # ground truth of I-SKELETON / I-TARGETS / I-CPDAG
    aug_graph_path = f'../../datasets/experiment/original/{exp_name}/aug_graphs/{benchmark_name}_aug_graph.txt'
    raw_dataset_path = f'../../datasets/experiment/original/{exp_name}/raw_samples/{benchmark_name}_raw_dataset.pkl'
    int_targets_path = f'../../datasets/experiment/original/{exp_name}/int_targets/{benchmark_name}_int_targets.pkl'
    targ_I_SKELETON, targ_I_TARGETS, targ_I_CPDAG = get_compared_components(aug_graph_path, intervention_size, real=True)
    
    try:
        know_pred_graph_path = f'../../baselines/exps_of_result/ut-igsp/{exp_name}/know/{benchmark_name}_aug_graph.txt'
        unknow_pred_graph_path = f'../../baselines/exps_of_result/ut-igsp/{exp_name}/unknow/{benchmark_name}_aug_graph.txt'

        with open(raw_dataset_path, 'rb') as f:
            data_list = pickle.load(f)
        with open(int_targets_path, 'rb') as f:
            know_targets_list = pickle.load(f)
        
        obs_samples, iv_samples_list = data_list[0], data_list[1:]
        nodes = set(range(obs_samples.shape[1]))
        obs_suffstat = partial_correlation_suffstat(obs_samples)
        ci_tester = MemoizedCI_Tester(partial_correlation_test, obs_suffstat, alpha=1e-3)
        invariance_suffstat = gauss_invariance_suffstat(obs_samples, iv_samples_list)
        invariance_tester = MemoizedInvarianceTester(gauss_invariance_test, invariance_suffstat, alpha=1e-3)
        
        unknow_setting_list = [dict(known_interventions=[]) for _ in know_targets_list[1:]]
        know_setting_list = [dict(known_interventions=targets) for targets in know_targets_list[1:]]
        
        pred_I_CPDAG_unknow, pred_I_TARGETS_unknow = unknown_target_igsp(unknow_setting_list, nodes, ci_tester, invariance_tester)
        pred_I_CPDAG_know, _ = unknown_target_igsp(know_setting_list, nodes, ci_tester, invariance_tester)
        
        pred_I_CPDAG_unknow, pred_I_CPDAG_know = pred_I_CPDAG_unknow.to_amat()[0], pred_I_CPDAG_know.to_amat()[0]

        with open(unknow_pred_graph_path, 'wb') as f:
            np.savetxt(f, pred_I_CPDAG_unknow, fmt='%i')

        with open(know_pred_graph_path, 'wb') as f:
            np.savetxt(f, pred_I_CPDAG_know, fmt='%i')
        
        pred_I_SKELETON_unknow = get_skeleton(pred_I_CPDAG_unknow)
        mt_skeleton_unknow = metric_skeleton_level(pred_I_SKELETON_unknow, targ_I_SKELETON)
        mt_target_unknow = metric_target_level_for_utigsp(pred_I_TARGETS_unknow, list(map(set, know_targets_list[1:])))
        mt_cpdag_unknow = metric_cpdag_level(pred_I_CPDAG_unknow, targ_I_CPDAG)
        print(f'unknow performance: \n skeleton:{mt_skeleton_unknow} \n targets:{mt_target_unknow} \n cpdag:{mt_cpdag_unknow} \n')

        pred_I_SKELETON_know = get_skeleton(pred_I_CPDAG_know)
        mt_skeleton_know = metric_skeleton_level(pred_I_SKELETON_know, targ_I_SKELETON)
        mt_cpdag_know = metric_cpdag_level(pred_I_CPDAG_know, targ_I_CPDAG)
        print(f'know performance: \n skeleton:{mt_skeleton_know} \n  cpdag:{mt_cpdag_know} \n')
    except:
        print(f'pass {benchmark_name}\n')

In [ ]:
exp_name = 'exp_2_5'
intervention_size_list = [1, 1, 2, 2, 4, 5, 6, 7, 7, 10, 11, 14, 15, 22]
os.makedirs(f'../../exps_of_result/ut-igsp/{exp_name}/know', exist_ok=True)
os.makedirs(f'../../baselines/exps_of_result/ut-igsp/{exp_name}/unknow', exist_ok=True)


for idx, benchmark_name in enumerate(['01earthquake', '02survey', '03asia', '04sachs',  '05child', '06insurance', '07water', '08mildew', '09alarm', '10barley', '11hailfinder', '12hepar2', '13win95pts', '14pathfinder']):
    if idx==0 or idx>=8:
        continue
    print(f"{'-'*60} {benchmark_name} {'-'*60} \n")
    intervention_size = intervention_size_list[idx]
    # ground truth of I-SKELETON / I-TARGETS / I-CPDAG
    aug_graph_path = f'../../datasets/experiment/original/{exp_name}/aug_graphs/{benchmark_name}_aug_graph.txt'
    raw_dataset_path = f'../../datasets/experiment/original/{exp_name}/raw_samples/{benchmark_name}_raw_dataset.pkl'
    int_targets_path = f'../../datasets/experiment/original/{exp_name}/int_targets/{benchmark_name}_int_targets.pkl'
    targ_I_SKELETON, targ_I_TARGETS, targ_I_CPDAG = get_compared_components(aug_graph_path, intervention_size, real=True)
    
    try:
        know_pred_graph_path = f'../../baselines/exps_of_result/ut-igsp/{exp_name}/know/{benchmark_name}_aug_graph.txt'
        unknow_pred_graph_path = f'../../baselines/exps_of_result/ut-igsp/{exp_name}/unknow/{benchmark_name}_aug_graph.txt'

        with open(raw_dataset_path, 'rb') as f:
            data_list = pickle.load(f)
        with open(int_targets_path, 'rb') as f:
            know_targets_list = pickle.load(f)
        
        obs_samples, iv_samples_list = data_list[0], data_list[1:]
        nodes = set(range(obs_samples.shape[1]))
        obs_suffstat = partial_correlation_suffstat(obs_samples)
        ci_tester = MemoizedCI_Tester(partial_correlation_test, obs_suffstat, alpha=1e-3)
        invariance_suffstat = gauss_invariance_suffstat(obs_samples, iv_samples_list)
        invariance_tester = MemoizedInvarianceTester(gauss_invariance_test, invariance_suffstat, alpha=1e-3)
        
        unknow_setting_list = [dict(known_interventions=[]) for _ in know_targets_list[1:]]
        know_setting_list = [dict(known_interventions=targets) for targets in know_targets_list[1:]]
        
        pred_I_CPDAG_unknow, pred_I_TARGETS_unknow = unknown_target_igsp(unknow_setting_list, nodes, ci_tester, invariance_tester)
        pred_I_CPDAG_know, _ = unknown_target_igsp(know_setting_list, nodes, ci_tester, invariance_tester)
        
        pred_I_CPDAG_unknow, pred_I_CPDAG_know = pred_I_CPDAG_unknow.to_amat()[0], pred_I_CPDAG_know.to_amat()[0]

        with open(unknow_pred_graph_path, 'wb') as f:
            np.savetxt(f, pred_I_CPDAG_unknow, fmt='%i')

        with open(know_pred_graph_path, 'wb') as f:
            np.savetxt(f, pred_I_CPDAG_know, fmt='%i')
        
        pred_I_SKELETON_unknow = get_skeleton(pred_I_CPDAG_unknow)
        mt_skeleton_unknow = metric_skeleton_level(pred_I_SKELETON_unknow, targ_I_SKELETON)
        mt_target_unknow = metric_target_level_for_utigsp(pred_I_TARGETS_unknow, list(map(set, know_targets_list[1:])))
        mt_cpdag_unknow = metric_cpdag_level(pred_I_CPDAG_unknow, targ_I_CPDAG)
        print(f'unknow performance: \n skeleton:{mt_skeleton_unknow} \n targets:{mt_target_unknow} \n cpdag:{mt_cpdag_unknow} \n')

        pred_I_SKELETON_know = get_skeleton(pred_I_CPDAG_know)
        mt_skeleton_know = metric_skeleton_level(pred_I_SKELETON_know, targ_I_SKELETON)
        mt_cpdag_know = metric_cpdag_level(pred_I_CPDAG_know, targ_I_CPDAG)
        print(f'know performance: \n skeleton:{mt_skeleton_know} \n  cpdag:{mt_cpdag_know} \n')
    except:
        print(f'pass {benchmark_name}\n')

In [ ]:
exp_name = 'exp_2_8'
intervention_size_list = [1, 1, 2, 2, 4, 5, 6, 7, 7, 10, 11, 14, 15, 22]
os.makedirs(f'../../exps_of_result/ut-igsp/{exp_name}/know', exist_ok=True)
os.makedirs(f'../../baselines/exps_of_result/ut-igsp/{exp_name}/unknow', exist_ok=True)


for idx, benchmark_name in enumerate(['01earthquake', '02survey', '03asia', '04sachs',  '05child', '06insurance', '07water', '08mildew', '09alarm', '10barley', '11hailfinder', '12hepar2', '13win95pts', '14pathfinder']):
    if idx==0 or idx>=8:
        continue
    print(f"{'-'*60} {benchmark_name} {'-'*60} \n")
    intervention_size = intervention_size_list[idx]
    # ground truth of I-SKELETON / I-TARGETS / I-CPDAG
    aug_graph_path = f'../../datasets/experiment/original/{exp_name}/aug_graphs/{benchmark_name}_aug_graph.txt'
    raw_dataset_path = f'../../datasets/experiment/original/{exp_name}/raw_samples/{benchmark_name}_raw_dataset.pkl'
    int_targets_path = f'../../datasets/experiment/original/{exp_name}/int_targets/{benchmark_name}_int_targets.pkl'
    targ_I_SKELETON, targ_I_TARGETS, targ_I_CPDAG = get_compared_components(aug_graph_path, intervention_size, real=True)
    
    try:
        know_pred_graph_path = f'../../baselines/exps_of_result/ut-igsp/{exp_name}/know/{benchmark_name}_aug_graph.txt'
        unknow_pred_graph_path = f'../../baselines/exps_of_result/ut-igsp/{exp_name}/unknow/{benchmark_name}_aug_graph.txt'

        with open(raw_dataset_path, 'rb') as f:
            data_list = pickle.load(f)
        with open(int_targets_path, 'rb') as f:
            know_targets_list = pickle.load(f)
        
        obs_samples, iv_samples_list = data_list[0], data_list[1:]
        nodes = set(range(obs_samples.shape[1]))
        obs_suffstat = partial_correlation_suffstat(obs_samples)
        ci_tester = MemoizedCI_Tester(partial_correlation_test, obs_suffstat, alpha=1e-3)
        invariance_suffstat = gauss_invariance_suffstat(obs_samples, iv_samples_list)
        invariance_tester = MemoizedInvarianceTester(gauss_invariance_test, invariance_suffstat, alpha=1e-3)
        
        unknow_setting_list = [dict(known_interventions=[]) for _ in know_targets_list[1:]]
        know_setting_list = [dict(known_interventions=targets) for targets in know_targets_list[1:]]
        
        pred_I_CPDAG_unknow, pred_I_TARGETS_unknow = unknown_target_igsp(unknow_setting_list, nodes, ci_tester, invariance_tester)
        pred_I_CPDAG_know, _ = unknown_target_igsp(know_setting_list, nodes, ci_tester, invariance_tester)
        
        pred_I_CPDAG_unknow, pred_I_CPDAG_know = pred_I_CPDAG_unknow.to_amat()[0], pred_I_CPDAG_know.to_amat()[0]

        with open(unknow_pred_graph_path, 'wb') as f:
            np.savetxt(f, pred_I_CPDAG_unknow, fmt='%i')

        with open(know_pred_graph_path, 'wb') as f:
            np.savetxt(f, pred_I_CPDAG_know, fmt='%i')
        
        pred_I_SKELETON_unknow = get_skeleton(pred_I_CPDAG_unknow)
        mt_skeleton_unknow = metric_skeleton_level(pred_I_SKELETON_unknow, targ_I_SKELETON)
        mt_target_unknow = metric_target_level_for_utigsp(pred_I_TARGETS_unknow, list(map(set, know_targets_list[1:])))
        mt_cpdag_unknow = metric_cpdag_level(pred_I_CPDAG_unknow, targ_I_CPDAG)
        print(f'unknow performance: \n skeleton:{mt_skeleton_unknow} \n targets:{mt_target_unknow} \n cpdag:{mt_cpdag_unknow} \n')

        pred_I_SKELETON_know = get_skeleton(pred_I_CPDAG_know)
        mt_skeleton_know = metric_skeleton_level(pred_I_SKELETON_know, targ_I_SKELETON)
        mt_cpdag_know = metric_cpdag_level(pred_I_CPDAG_know, targ_I_CPDAG)
        print(f'know performance: \n skeleton:{mt_skeleton_know} \n  cpdag:{mt_cpdag_know} \n')
    except:
        print(f'pass {benchmark_name}\n')

In [ ]:

intervention_size_list = [1, 1, 2, 2, 4, 5, 6, 7, 7, 10, 11, 14, 15, 22]


for exp_name in  ['exp_3_1', 'exp_3_2', 'exp_3_3', 'exp_3_4', 'exp_3_5']:
    os.makedirs(f'../../exps_of_result/ut-igsp/{exp_name}/know', exist_ok=True)
    os.makedirs(f'../../baselines/exps_of_result/ut-igsp/{exp_name}/unknow', exist_ok=True)
    for idx, benchmark_name in enumerate(['01earthquake', '02survey', '03asia', '04sachs',  '05child', '06insurance', '07water', '08mildew', '09alarm', '10barley', '11hailfinder', '12hepar2', '13win95pts', '14pathfinder']):
        if idx!=5:
            continue
        print(f"{'-'*60} {benchmark_name} {'-'*60} \n")
        if exp_name == 'exp_3_1':
            intervention_size = 0
        elif exp_name == 'exp_3_2':
            intervention_size = 8
        elif exp_name == 'exp_3_3':
            intervention_size = 12
        elif exp_name == 'exp_3_4':
            intervention_size = 16
        elif exp_name == 'exp_3_5':
            intervention_size = 20
        # ground truth of I-SKELETON / I-TARGETS / I-CPDAG
        aug_graph_path = f'../../datasets/experiment/original/{exp_name}/aug_graphs/{benchmark_name}_aug_graph.txt'
        raw_dataset_path = f'../../datasets/experiment/original/{exp_name}/raw_samples/{benchmark_name}_raw_dataset.pkl'
        int_targets_path = f'../../datasets/experiment/original/{exp_name}/int_targets/{benchmark_name}_int_targets.pkl'
        targ_I_SKELETON, targ_I_TARGETS, targ_I_CPDAG = get_compared_components(aug_graph_path, intervention_size, real=True)
        
        try:
            know_pred_graph_path = f'../../baselines/exps_of_result/ut-igsp/{exp_name}/know/{benchmark_name}_aug_graph.txt'
            unknow_pred_graph_path = f'../../baselines/exps_of_result/ut-igsp/{exp_name}/unknow/{benchmark_name}_aug_graph.txt'

            with open(raw_dataset_path, 'rb') as f:
                data_list = pickle.load(f)
            with open(int_targets_path, 'rb') as f:
                know_targets_list = pickle.load(f)
            
            obs_samples, iv_samples_list = data_list[0], data_list[1:]
            nodes = set(range(obs_samples.shape[1]))
            obs_suffstat = partial_correlation_suffstat(obs_samples)
            ci_tester = MemoizedCI_Tester(partial_correlation_test, obs_suffstat, alpha=1e-3)
            invariance_suffstat = gauss_invariance_suffstat(obs_samples, iv_samples_list)
            invariance_tester = MemoizedInvarianceTester(gauss_invariance_test, invariance_suffstat, alpha=1e-3)
            
            unknow_setting_list = [dict(known_interventions=[]) for _ in know_targets_list[1:]]
            know_setting_list = [dict(known_interventions=targets) for targets in know_targets_list[1:]]
            
            pred_I_CPDAG_unknow, pred_I_TARGETS_unknow = unknown_target_igsp(unknow_setting_list, nodes, ci_tester, invariance_tester)
            pred_I_CPDAG_know, _ = unknown_target_igsp(know_setting_list, nodes, ci_tester, invariance_tester)
            
            pred_I_CPDAG_unknow, pred_I_CPDAG_know = pred_I_CPDAG_unknow.to_amat()[0], pred_I_CPDAG_know.to_amat()[0]

            with open(unknow_pred_graph_path, 'wb') as f:
                np.savetxt(f, pred_I_CPDAG_unknow, fmt='%i')

            with open(know_pred_graph_path, 'wb') as f:
                np.savetxt(f, pred_I_CPDAG_know, fmt='%i')
            
            pred_I_SKELETON_unknow = get_skeleton(pred_I_CPDAG_unknow)
            mt_skeleton_unknow = metric_skeleton_level(pred_I_SKELETON_unknow, targ_I_SKELETON)
            mt_target_unknow = metric_target_level_for_utigsp(pred_I_TARGETS_unknow, list(map(set, know_targets_list[1:])))
            mt_cpdag_unknow = metric_cpdag_level(pred_I_CPDAG_unknow, targ_I_CPDAG)
            print(f'unknow performance: \n skeleton:{mt_skeleton_unknow} \n targets:{mt_target_unknow} \n cpdag:{mt_cpdag_unknow} \n')

            pred_I_SKELETON_know = get_skeleton(pred_I_CPDAG_know)
            mt_skeleton_know = metric_skeleton_level(pred_I_SKELETON_know, targ_I_SKELETON)
            mt_cpdag_know = metric_cpdag_level(pred_I_CPDAG_know, targ_I_CPDAG)
            print(f'know performance: \n skeleton:{mt_skeleton_know} \n  cpdag:{mt_cpdag_know} \n')
        except:
            print(f'pass {benchmark_name}\n')